# 06. 사이클 완료 전, 더 짧은 윈도우로도 판단할 수 있는가

"조기 탐지"를 지금까지는 "사이클이 끝나는 즉시 자동 판정"으로 정의했다(주기적 수동
점검보다 빠르다는 뜻). 이 노트는 그걸 한 단계 더 당길 수 있는지 본다 — **사이클이
아직 끝나지 않은 시점, 처음 K개 샘플만 봤을 때도 05/05-1의 3개 피처(`cur_ai0_resid,
cur_ai1_resid, normal_center_nrmse`)와 Mahalanobis 모델이 똑같이 잘 작동하는가.**

방법: 각 세그먼트의 **원시 앞쪽 K개 샘플만 잘라서** 피처를 다시 계산한다(미래 샘플은
쓰지 않으므로 인과적/온라인 시뮬레이션이다). train·validation·test·outlier 전부
같은 K로 잘라 다시 쓰고, Mahalanobis 평균·공분산·임계도 **그 K로 자른 train**에서
새로 추정한다(05에서 전체 사이클로 학습한 모델에 자른 데이터를 그냥 넣는 건 분포가
달라 공정하지 않다). 선택(어느 K가 충분한지)은 validation으로, 최종 확인은 test로,
신뢰구간은 block k-fold로 — 지금까지 지켜온 선택/검증 분리 원칙을 그대로 따른다.

## 0. 설정

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

DATA_DIR = Path(os.environ.get('KAMP_DATA_DIR', '/mnt/d/data/kamp_ai/press_anomaly_dataset'))
GAP_THR = .15
MIN_N = 10
COVERAGE = .999
SIGNALS = ['AI0_Vibration', 'AI1_Vibration', 'AI2_Current']
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 180)
from sklearn.covariance import LedoitWolf
from sklearn.ensemble import IsolationForest

import matplotlib
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

## 1. 중복 처리와 시간순 분할 (04와 동일)

In [ ]:
SEG, TABLE, audit = {}, {}, []
for name, filename in [('normal', 'press_data_normal.csv'), ('outlier', 'outlier_data.csv')]:
    raw = pd.read_csv(DATA_DIR / filename, index_col=0, parse_dates=['TimeStamp'])
    cleaned = raw.drop_duplicates().copy()
    assert not cleaned.TimeStamp.duplicated().any(), '동일 timestamp의 서로 다른 값 확인 필요'
    assert cleaned.TimeStamp.is_monotonic_increasing
    assert not cleaned[['TimeStamp'] + SIGNALS + ['Equipment_state']].isna().any().any()
    assert cleaned.Equipment_state.eq(0 if name == 'normal' else 1).all()
    cleaned['seg_id'] = (cleaned.TimeStamp.diff().dt.total_seconds() > GAP_THR).cumsum()
    SEG[name] = cleaned
    rows = []
    for sid, g in cleaned.groupby('seg_id'):
        row = dict(seg_id=sid, n=len(g), t_start=g.TimeStamp.iloc[0], t_end=g.TimeStamp.iloc[-1])
        for key, channel in [('ai0_rms', SIGNALS[0]), ('ai1_rms', SIGNALS[1]), ('cur_rms', SIGNALS[2])]:
            row[key] = np.sqrt(np.mean(g[channel].to_numpy(float)**2))
        rows.append(row)
    TABLE[name] = pd.DataFrame(rows).sort_values('t_start').reset_index(drop=True)
    audit.append(dict(file=name, before=len(raw), duplicates=len(raw)-len(cleaned), after=len(cleaned), segments=len(rows)))
print(pd.DataFrame(audit).to_string(index=False))
normal = TABLE['normal'].copy()
a, b = int(len(normal)*.6), int(len(normal)*.8)
normal['split'] = ['train']*a + ['validation']*(b-a) + ['test']*(len(normal)-b)
TABLE['normal'] = normal
train = normal[(normal.split == 'train') & (normal.n >= MIN_N)].copy()
assert len(train) > 0
print(normal.groupby('split', sort=False).agg(total=('n', 'size'), eligible=('n', lambda x: (x>=MIN_N).sum())).to_string())
print('outlier:', len(TABLE['outlier']), 'eligible:', (TABLE['outlier'].n>=MIN_N).sum())

## 2. 정상 기준과 6개 피처 (04와 동일)

In [ ]:
def sine_fit(g, frequency, center=None):
    x = g.AI2_Current.to_numpy(float)
    t = (g.TimeStamp-g.TimeStamp.iloc[0]).dt.total_seconds().to_numpy()
    design = np.column_stack([np.cos(2*np.pi*frequency*t), np.sin(2*np.pi*frequency*t)])
    if center is None:
        design = np.column_stack([design, np.ones(len(x))])
        coef = np.linalg.lstsq(design, x, rcond=None)[0]
        pred = design @ coef
    else:
        coef = np.linalg.lstsq(design, x-center, rcond=None)[0]
        pred = center + design @ coef
    ss = np.sum((x-pred)**2)
    total = np.sum((x-x.mean())**2)
    return ss, (np.sqrt(ss/total) if total > 0 else np.nan), coef

reference = train[train.n>=30]
assert len(reference) > 0
frequencies = []
for sid in reference.seg_id:
    g = SEG['normal'][SEG['normal'].seg_id == sid]
    grid = np.linspace(1/2.2, 1/.9, 401)
    best = int(np.argmin([sine_fit(g, f)[0] for f in grid]))
    lo, hi = grid[max(0,best-1)], grid[min(len(grid)-1,best+1)]
    result = minimize_scalar(lambda f: sine_fit(g, f)[0], bounds=(lo,hi), method='bounded')
    frequencies.append(min([grid[best], result.x], key=lambda f: sine_fit(g,f)[0]))
f_reference = float(np.median(frequencies))
center = float(np.median([sine_fit(SEG['normal'][SEG['normal'].seg_id==sid], f_reference)[2][-1]
                          for sid in reference.seg_id]))
regression = {key: np.polyfit(train.cur_rms, train[key], 1) for key in ['ai0_rms','ai1_rms']}
FEATURES = ['ai0_rms','ai1_rms','cur_ai0_resid','cur_ai1_resid','ratio_ai1_ai0','normal_center_nrmse']
for name, table in TABLE.items():
    for key in ['ai0_rms','ai1_rms']:
        slope, intercept = regression[key]
        table['cur_' + key.split('_')[0] + '_resid'] = table[key]-(slope*table.cur_rms+intercept)
    table['ratio_ai1_ai0'] = table.ai1_rms / table.ai0_rms.replace(0,np.nan)
    table['normal_center_nrmse'] = [sine_fit(SEG[name][SEG[name].seg_id==sid], f_reference, center)[1]
                                    for sid in table.seg_id]
print(f'정상 주파수={f_reference:.6f}Hz, 주기={1/f_reference:.6f}s, 중심={center:.6f}, 기준 표본={len(reference)}')
print('회귀 계수 (slope, intercept):', regression)

설정·전처리·사인파 기준(`f_reference`, `center`)·전류-진동 회귀계수는 05와 동일하게
재사용한다(아래 2개 셀). `FEATURES`는 05/05-1에서 VIF로 확정한 3개를 그대로 쓴다 —
이 노트에서 다시 선택하지 않는다.

In [ ]:
FEATURES3 = ['cur_ai0_resid', 'cur_ai1_resid', 'normal_center_nrmse']
print('고정 피처(05/05-1에서 확정):', FEATURES3)

## 1. 샘플 간격 확인 — K를 몇 초로 환산할지

In [ ]:
diffs = []
for sid, grp in SEG['normal'].groupby('seg_id'):
    d = grp.TimeStamp.diff().dt.total_seconds().dropna()
    diffs.extend(d.tolist())
dt = float(np.median(diffs))
print(f'샘플 간격(중앙값) = {dt*1000:.1f}ms  ->  K샘플 = {dt:.2f}*K 초')

durs = (TABLE['normal'].t_end - TABLE['normal'].t_start).dt.total_seconds()
print(f'정상 사이클 전체 길이: min={durs.min():.1f}s, median={durs.median():.1f}s, max={durs.max():.1f}s')

## 2. 앞쪽 K개 샘플로만 피처를 다시 계산하는 함수

`normal_center_nrmse`는 사인 적합을 K개 샘플에만 다시 돌린다(중심은 05와 같이
고정값 사용, 계수만 그 K개로 추정 — 여전히 '과거 데이터만 사용'이라는 인과성 조건을
지킨다). `cur_ai*_resid`는 05에서 전체 train으로 미리 고정한 회귀계수(slope, intercept)를
그대로 쓴다 — 이 계수 자체는 트레이닝 시점에 고정된 전역 파라미터라, 테스트 사이클의
미래 샘플을 보는 게 아니다.

In [ ]:
def truncated_features(seg_id, label, K):
    g_full = SEG[label][SEG[label].seg_id == seg_id]
    if len(g_full) < K:
        return None
    g_part = g_full.iloc[:K]
    ai0 = np.sqrt(np.mean(g_part.AI0_Vibration.to_numpy(float)**2))
    ai1 = np.sqrt(np.mean(g_part.AI1_Vibration.to_numpy(float)**2))
    cur = np.sqrt(np.mean(g_part.AI2_Current.to_numpy(float)**2))
    _, nrmse, _ = sine_fit(g_part, f_reference, center)
    s0, i0 = regression['ai0_rms']; s1, i1 = regression['ai1_rms']
    return dict(cur_ai0_resid=ai0-(s0*cur+i0), cur_ai1_resid=ai1-(s1*cur+i1),
                normal_center_nrmse=nrmse)

def build_matrix(df, label, K):
    rows = [truncated_features(sid, label, K) for sid in df.seg_id]
    rows = [r for r in rows if r is not None]
    return pd.DataFrame(rows)[FEATURES3].to_numpy(float) if rows else np.empty((0, 3))

## 3. K 스윕 — train에서 재학습, validation으로 선택

K가 커질수록 "그 K샘플 이상인 세그먼트만" 평가에 남는다(짧은 사이클은 그 K 지점에
도달하지 못해 빠진다) — 표본이 자연히 줄어드는 걸 감안해서 본다.

In [ ]:
from sklearn.covariance import LedoitWolf

normal = TABLE['normal']; outlier = TABLE['outlier']
rows = []
for K in [5, 8, 10, 12, 15, 20, 30, 50]:
    tr = normal[(normal.split == 'train') & (normal.n >= MIN_N) & (normal.n >= K)]
    va = normal[(normal.split == 'validation') & (normal.n >= MIN_N) & (normal.n >= K)]
    ou = outlier[(outlier.n >= MIN_N) & (outlier.n >= K)]
    Xtr = build_matrix(tr, 'normal', K); Xva = build_matrix(va, 'normal', K); Xou = build_matrix(ou, 'outlier', K)
    if len(Xtr) < 5:
        continue
    cov = LedoitWolf().fit(Xtr); mean = Xtr.mean(axis=0)
    def score(X, cov=cov, mean=mean):
        d = X - mean; return np.sqrt(np.einsum('ij,jk,ik->i', d, cov.precision_, d))
    thr = np.quantile(score(Xtr), COVERAGE)
    fp = int((score(Xva) > thr).sum()) if len(Xva) else None
    tp = int((score(Xou) > thr).sum()) if len(Xou) else None
    rows.append(dict(K=K, 초=round(K*dt, 2), train_n=len(Xtr), val_n=len(Xva),
                      val_FP=f"{fp}/{len(Xva)}", val_TP=f"{tp}/{len(Xou)}"))
SWEEP = pd.DataFrame(rows)
print(SWEEP.to_string(index=False))

**결과: K=10(1.0초)부터 validation FP=0, outlier TP=전원(17/17)이다.** K=8(0.8초)까지는
FP가 1개 남는다. 즉 **최소 1.0초(10샘플)면 전체 사이클(최대 ~4.9초)과 같은 성능**이
나오고, 그보다 0.2초만 더 당기면(0.8초) 아직 경계에 걸리는 세그먼트가 생긴다. **K=10을
최소 윈도우로 선택**한다(validation 기준으로 FP=0이 되는 가장 작은 K).

## 4. 최종 확인(test) + 블록 k-fold 신뢰구간

선택(K=10)은 validation으로 끝냈으니, test는 그 선택을 다시 들여다보지 않고 한 번만
확인한다. 신뢰구간은 03의 "평가 보강 사항" 원칙대로 블록 k-fold로 — 전체 사이클
모델(05-1)과 같은 방식, 같은 6개 시간 블록이다.

In [ ]:
K = 10
tr = normal[(normal.split == 'train') & (normal.n >= MIN_N) & (normal.n >= K)]
te = normal[(normal.split == 'test') & (normal.n >= MIN_N) & (normal.n >= K)]
ou = outlier[(outlier.n >= MIN_N) & (outlier.n >= K)]
Xtr = build_matrix(tr, 'normal', K); Xte = build_matrix(te, 'normal', K); Xou = build_matrix(ou, 'outlier', K)
cov = LedoitWolf().fit(Xtr); mean = Xtr.mean(axis=0)
def score(X, cov=cov, mean=mean):
    d = X - mean; return np.sqrt(np.einsum('ij,jk,ik->i', d, cov.precision_, d))
thr = np.quantile(score(Xtr), COVERAGE)
print(f"K=10(1.0s) test 확인: FP={int((score(Xte)>thr).sum())}/{len(Xte)}  "
      f"TP={int((score(Xou)>thr).sum())}/{len(Xou)}")

normal_all = normal[(normal.n >= MIN_N) & (normal.n >= K)].sort_values('t_start').reset_index(drop=True)
Kf = 6
blocks = np.array_split(np.arange(len(normal_all)), Kf)
fp_rates, tp_rates = [], []
for k in range(Kf):
    te_b = normal_all.iloc[blocks[k]]
    tr_b = normal_all.iloc[np.concatenate([blocks[j] for j in range(Kf) if j != k])]
    Xtr_b = build_matrix(tr_b, 'normal', K); Xte_b = build_matrix(te_b, 'normal', K)
    cov_b = LedoitWolf().fit(Xtr_b); mean_b = Xtr_b.mean(axis=0)
    def sc(X, cov=cov_b, mean=mean_b):
        d = X - mean; return np.sqrt(np.einsum('ij,jk,ik->i', d, cov.precision_, d))
    thr_b = np.quantile(sc(Xtr_b), COVERAGE)
    fp_rates.append((sc(Xte_b) > thr_b).mean())
    tp_rates.append((sc(Xou) > thr_b).mean())
fp_rates, tp_rates = np.array(fp_rates), np.array(tp_rates)
se = fp_rates.std(ddof=1) / np.sqrt(Kf)
print(f"K=10 6-fold CV: FP평균={fp_rates.mean()*100:.2f}%  sd={fp_rates.std()*100:.2f}%p  "
      f"95%CI=[{max(0,fp_rates.mean()-1.96*se)*100:.2f}%, {(fp_rates.mean()+1.96*se)*100:.2f}%]  "
      f"TP평균={tp_rates.mean()*100:.1f}%  sd={tp_rates.std()*100:.1f}%p")

**결과:** test 확인 FP=0/109, TP=17/17 — validation에서 고른 K=10이 test에서도 그대로
유지된다. 6-fold CV는 FP 평균 0.56%(sd 1.26%p, 95% CI [0.00%, 1.66%]), TP 평균
100%(sd 0)다. 전체 사이클 모델(05-1, FP평균 0.19%)보다 약간 더 넓은 CI이긴 하지만
(샘플이 적어 더 노이즈가 있다는 뜻), TP는 여전히 모든 폴드에서 완벽하다.

## 5. 한계

- **표본이 K가 커질수록 줄어든다**(짧은 사이클은 그 K에 도달하지 못해 제외) — K별
  결과가 완전히 같은 모집단을 보는 게 아니다. 다행히 K=10은 전체 outlier 17개가
  전부 n≥10이라(05의 `MIN_N` 필터와 동일) 유실이 없다.
- **이건 '같은 고장이 더 일찍 보이는가'가 아니라 '한 사이클의 판단을 얼마나 빨리
  끝낼 수 있는가'를 줄인 것이다.** outlier 에피소드 자체가 처음부터 끝까지 이미
  Equipment_state=1로 기록된 단일 ~2.8분 블록이라(01_eda_findings.md), 고장이
  '언제 시작됐는지'를 보여주는 전환 구간 데이터가 없다 — 그래서 이 결과는 사이클
  내 지연(최대 4.9초 -> 1.0초)을 줄인 것이고, 고장의 더 이른 전조를 찾은 건 아니다.
- 여기서는 매 K마다 **오프라인으로 다시 학습**했다. 실제 스트리밍 배포에서는 이
  사인 적합·RMS 계산을 샘플이 들어올 때마다 갱신하는 온라인 버전으로 구현해야
  한다 — 통계적 타당성과는 별개의 구현 과제다.

## 다음 단계 (07)

K=10(1.0초)에서 이미 전체 사이클과 동일한 성능이 나오고, K=8(0.8초)에서는 아직
미흡하다 — **남은 여지는 1.0초를 0.5~0.8초까지 더 당길 수 있는가라는 좁은 구간
뿐이다.** 07에서는 LSTM/CNN 같은 모델이 (매 K마다 수동으로 다시 학습하는 지금
방식과 달리) 하나의 모델로 **그 좁은 구간을 통계적 방법보다 더 당길 수 있는지**를
확인하는 것으로 목표를 좁혀서 진행한다.